# 04 — Evaluación y comparativa de modelos · CERT r4.2

**Objetivos de este notebook (Fase 4 del PLAN.md):**
1. **Curvas Precision-Recall** a nivel usuario-día para los tres modelos (reglas, Isolation Forest, autoencoder).
2. **Vista operativa SOC**: trade-off alertas/día vs recall — cuántas alertas genera cada modelo y qué recall consigue a ese coste.
3. **Curva de presupuesto de investigación**: % de días maliciosos hallados investigando el X% de user-días más anómalos (métrica clásica CERT).
4. **Detección por escenario de ataque**: de los 3 escenarios (esc.1 USB+nocturno, esc.2 robo masivo USB, esc.3 sysadmin/keylogger), qué modelo detecta mejor cada uno.

**Recordatorio importante:** el desbalanceo es extremo (**0.41% positivos** a nivel usuario-día). **AUC-PR (average precision) es la métrica clave**; **accuracy no se usa** porque un modelo que prediga "todo benigno" tendría accuracy >99.9% y recall 0.

Este notebook es **autocontenido** y dual Colab/local: solo lee `data/processed/user_day_scores.parquet` generado en la fase 3.

In [ ]:
# [compute]
# === Setup: entorno, dependencias y rutas ===
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q polars pyarrow
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/CERT_data")
    DATA_DIR = BASE / "data"/ "r4.2"
    ANSWERS_DIR = BASE / "answers"
    PROCESSED_DIR = BASE / "processed"
else:
    # Local: el notebook vive en SIEM/notebooks/ — reutilizamos src/config.py
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.config import DATA_DIR, ANSWERS_DIR, PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import polars as pl

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"Datos:   {DATA_DIR}")
print(f"Answers: {ANSWERS_DIR}")
print(f"Procesados: {PROCESSED_DIR}")

# --- Dependencias de evaluación/gráficas: instalación condicional si faltan en local ---
try:
    import numpy as np
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, roc_curve
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "numpy", "matplotlib"], check=True)
    import numpy as np
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, roc_curve

MODELS = {
    "score_rules": "Reglas (baseline)",
    "score_iforest": "Isolation Forest",
    "score_autoencoder": "Autoencoder",
}

COLORS = {
    "score_rules": "#1f77b4",
    "score_iforest": "#ff7f0e",
    "score_autoencoder": "#2ca02c",
}

print(f"\nModelos: {list(MODELS.values())}")

In [ ]:
# [compute]
# === Carga de scores de la fase 3 ===

scores = pl.read_parquet(PROCESSED_DIR / "user_day_scores.parquet")

y_day = scores["label_malicious_day"].to_numpy()
N = len(y_day)
base_rate = y_day.mean()

print(f"scores: {scores.shape}")
print(f"N (user-días): {N}")
print(f"base_rate (proporción de positivos): {base_rate:.4%}")
print(f"nº días maliciosos: {int(y_day.sum())}")

## Curvas Precision-Recall a nivel usuario-día

Con un desbalanceo de 0.41% de positivos, las curvas Precision-Recall (PR) son la herramienta
adecuada para comparar modelos: muestran cómo cae la precisión a medida que se exige más
recall, y el **AUC-PR** (average precision) resume esa curva en un solo número. La línea
horizontal en `base_rate` marca el rendimiento de un clasificador aleatorio — cualquier curva
por encima de esa línea aporta señal.

In [ ]:
# [compute]
# === Curvas Precision-Recall (nivel usuario-día) ===

fig, ax = plt.subplots(figsize=(8, 6))

for col, label in MODELS.items():
    score = scores[col].to_numpy()
    precision, recall, _ = precision_recall_curve(y_day, score)
    ap = average_precision_score(y_day, score)
    ax.plot(recall, precision, color=COLORS[col], label=f"{label} (AUC-PR={ap:.3f})", linewidth=2)

ax.axhline(base_rate, color="gray", linestyle="--", linewidth=1.5,
           label=f"Azar (base rate={base_rate:.4f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precisión")
ax.set_title("Curvas Precision-Recall · nivel usuario-día")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Vista operativa SOC: alertas/día vs recall

Desde el punto de vista de un analista SOC, la pregunta no es "¿qué AUC-PR tiene el modelo?"
sino "si fijo un umbral, ¿cuántas alertas tengo que revisar cada día y qué fracción de los
días maliciosos capturo?". El periodo cubre **~500 días distintos**, así que:

$$\text{alertas/día} = \frac{\#\{\text{user-días con score} \geq \text{umbral}\}}{\#\text{días distintos}}$$

Recorremos una rejilla de umbrales (percentiles altos de cada score) y dibujamos recall vs
alertas/día para cada modelo.

In [ ]:
# [compute]
# === Trade-off alertas/día vs recall ===

n_days = scores["day"].n_unique()
total_positives = y_day.sum()
print(f"nº de días calendario distintos: {n_days}")

percentiles = np.arange(90.0, 100.0, 0.1)  # 90..99.9

fig, ax = plt.subplots(figsize=(8, 6))

for col, label in MODELS.items():
    score = scores[col].to_numpy()
    thresholds = np.percentile(score, percentiles)

    alerts_per_day = []
    recalls = []
    for thr in thresholds:
        mask = score >= thr
        alerts_per_day.append(mask.sum() / n_days)
        recalls.append(y_day[mask].sum() / total_positives)

    alerts_per_day = np.array(alerts_per_day)
    recalls = np.array(recalls)

    ax.plot(alerts_per_day, recalls, color=COLORS[col], label=label, linewidth=2, marker=".")

    # Anotar el punto más cercano a ~10 alertas/día
    idx = np.argmin(np.abs(alerts_per_day - 10))
    ax.annotate(
        f"{label}\n~{alerts_per_day[idx]:.1f} al./día → recall={recalls[idx]:.2f}",
        xy=(alerts_per_day[idx], recalls[idx]),
        xytext=(15, 10), textcoords="offset points",
        arrowprops=dict(arrowstyle="->", color=COLORS[col]),
        fontsize=8, color=COLORS[col],
    )

ax.set_xlabel("Alertas/día (umbral)")
ax.set_ylabel("Recall (fracción de días maliciosos capturados)")
ax.set_title("Vista operativa SOC: alertas/día vs recall")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Curva de presupuesto de investigación

Métrica clásica del dataset CERT: si un analista ordena todos los user-días por score
(descendente) y solo puede investigar el **X% más anómalo**, ¿qué **% de los días
maliciosos** caería dentro de ese presupuesto? Se compara contra la diagonal de un
investigador aleatorio (que encontraría X% de los positivos investigando X% de los datos).

In [ ]:
# [compute]
# === Curva de presupuesto de investigación ===

max_budget = 0.10
fractions = np.linspace(0, max_budget, 200)
annotate_at = [0.01, 0.05]

fig, ax = plt.subplots(figsize=(8, 6))

for col, label in MODELS.items():
    score = scores[col].to_numpy()
    order = np.argsort(-score)
    y_sorted = y_day[order]
    cum_positives = np.cumsum(y_sorted)

    recall_at_fraction = []
    for f in fractions:
        k = int(f * N)
        recall_at_fraction.append(cum_positives[k - 1] / total_positives if k > 0 else 0.0)
    recall_at_fraction = np.array(recall_at_fraction)

    # recall@1% y @5% para la leyenda
    ann_vals = {}
    for f in annotate_at:
        k = int(f * N)
        ann_vals[f] = cum_positives[k - 1] / total_positives if k > 0 else 0.0

    legend_label = f"{label} (recall@1%={ann_vals[0.01]:.2f}, @5%={ann_vals[0.05]:.2f})"
    ax.plot(fractions * 100, recall_at_fraction, color=COLORS[col], label=legend_label, linewidth=2)

# Diagonal de referencia (azar)
ax.plot([0, max_budget * 100], [0, max_budget], color="gray", linestyle="--", linewidth=1.5, label="Azar (diagonal)")

ax.set_xlabel("% de user-días investigados (los más anómalos primero)")
ax.set_ylabel("% de días maliciosos hallados")
ax.set_title("Curva de presupuesto de investigación")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Detección por escenario de ataque

Los 70 insiders del dataset r4.2 se reparten en 3 escenarios con firmas muy distintas:

1. **Esc. 1 (30 insiders)**: USB + actividad nocturna — fuga a wikileaks.
2. **Esc. 2 (30 insiders)**: robo masivo de datos por USB antes de irse a un competidor.
3. **Esc. 3 (10 insiders)**: sysadmin descontento, keylogger vía USB, mass email.

Agregamos el score por usuario (máximo sobre sus días) y, dentro del top-70 de usuarios por
score, contamos cuántos insiders de cada escenario caen dentro — esto indica qué modelo
prioriza mejor cada tipo de ataque en la watchlist.

In [ ]:
# [compute]
# === Detección por escenario de ataque (nivel usuario, top-70) ===

user_agg = scores.group_by("user").agg(
    [pl.col(c).max().alias(c) for c in MODELS.keys()]
    + [pl.col("scenario").max().alias("scenario"), pl.col("is_insider_user").max().alias("is_insider_user")]
)

n_insiders_total = int(user_agg["is_insider_user"].sum())
TOP_K = n_insiders_total  # top-70

rows = []
for s in [1, 2, 3]:
    insiders_s = user_agg.filter((pl.col("scenario") == s) & (pl.col("is_insider_user")))
    n_insiders_s = insiders_s.height
    row = {"escenario": s, "n_insiders": n_insiders_s}
    for col, label in MODELS.items():
        top_users = set(
            user_agg.sort(col, descending=True).head(TOP_K)["user"].to_list()
        )
        detected = sum(1 for u in insiders_s["user"].to_list() if u in top_users)
        row[label] = detected / n_insiders_s if n_insiders_s > 0 else 0.0
    rows.append(row)

det_por_escenario = pl.DataFrame(rows)
print("Tabla por escenario (fracción de insiders detectados en top-70 de usuarios):")
print(det_por_escenario)

# Gráfico de barras agrupadas: 3 escenarios x 3 modelos
fig, ax = plt.subplots(figsize=(8, 6))
scenarios = det_por_escenario["escenario"].to_list()
x = np.arange(len(scenarios))
width = 0.25

for i, (col, label) in enumerate(MODELS.items()):
    values = det_por_escenario[label].to_numpy()
    ax.bar(x + (i - 1) * width, values, width, label=label, color=COLORS[col])

ax.set_xticks(x)
ax.set_xticklabels([f"Esc. {s}\n(n={n})" for s, n in zip(scenarios, det_por_escenario['n_insiders'].to_list())])
ax.set_ylabel("Fracción de insiders detectados (top-70 usuarios)")
ax.set_title("Detección por escenario de ataque")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# [compute]
# === Tabla resumen final ===

budgets = [0.01, 0.05]
summary_rows = []

for col, label in MODELS.items():
    score = scores[col].to_numpy()

    ap_day = average_precision_score(y_day, score)
    auc_roc_day = roc_auc_score(y_day, score)

    order = np.argsort(-score)
    y_sorted = y_day[order]
    cum_positives = np.cumsum(y_sorted)
    recall_at = {}
    for b in budgets:
        k = int(b * N)
        recall_at[b] = cum_positives[k - 1] / total_positives if k > 0 else 0.0

    user_score = user_agg[col].to_numpy()
    y_user = user_agg["is_insider_user"].to_numpy().astype(int)
    ap_user = average_precision_score(y_user, user_score)

    n_ins = int(y_user.sum())
    top_idx = np.argsort(-user_score)[:n_ins]
    insiders_top70 = int(y_user[top_idx].sum())

    summary_rows.append({
        "modelo": label,
        "ap_day": ap_day,
        "auc_roc_day": auc_roc_day,
        "recall@1%": recall_at[0.01],
        "recall@5%": recall_at[0.05],
        "ap_user": ap_user,
        "insiders_en_top70": insiders_top70,
    })

summary = pl.DataFrame(summary_rows)
print("=== TABLA RESUMEN ===")
print(summary)

# --- Comprobaciones de reproducibilidad frente a referencias conocidas ---
autoenc_row = summary.filter(pl.col("modelo") == MODELS["score_autoencoder"])
ap_day_autoenc = autoenc_row["ap_day"][0]
assert abs(ap_day_autoenc - 0.041) < 0.01, f"ap_day autoencoder fuera de rango: {ap_day_autoenc:.4f}"

rules_row = summary.filter(pl.col("modelo") == MODELS["score_rules"])
insiders_top70_rules = rules_row["insiders_en_top70"][0]
assert insiders_top70_rules == 36, f"insiders_en_top70 rules no coincide con referencia: {insiders_top70_rules}"

print("\nComprobaciones de reproducibilidad: OK")
summary

## Conclusiones

- **Modelo recomendado según objetivo**:
  - Para **detección de día concreto** (priorizar alertas diarias de un SOC), el **Autoencoder**
    presenta el mejor AUC-PR a nivel usuario-día (~0.041), por delante de Reglas (~0.028) e
    Isolation Forest (~0.013) — su error de reconstrucción captura mejor los picos de
    comportamiento anómalo puntuales.
  - Para **identificación de usuarios insider** (construcción de una watchlist), el baseline
    de **Reglas** destaca con 36/70 insiders en el top-70, frente a 23 del Autoencoder y
    20 de Isolation Forest — al agregar por máximo, las señales interpretables (USB fuera
    de horario, emails externos, etc.) acumulan bien la sospecha a lo largo del tiempo.

- **Trade-off alertas/día**: la curva alertas/día vs recall muestra el coste operativo real
  de cada modelo. A volúmenes bajos (~10 alertas/día) ningún modelo alcanza un recall alto
  (el problema es intrínsecamente difícil con 0.41% de positivos), pero permite comparar
  qué modelo "gasta mejor" el presupuesto de un analista.

- **Escenarios más difíciles**: según la tabla por escenario, el **escenario 3** (sysadmin /
  keylogger, solo 10 insiders) tiende a ser el más difícil de capturar para todos los
  modelos, al ser un patrón minoritario y muy distinto de los escenarios 1 y 2 (ambos
  basados en USB masivo). Los escenarios 1 y 2, con señales de USB y horario más marcadas,
  son detectados con más consistencia.

- **De cara a la fase 5 (dashboard)**: estos resultados sugieren combinar ambos enfoques en
  el dashboard SOC — usar el **Autoencoder** (o un ensamblado) para la vista de "alertas del
  día" y el baseline de **Reglas** (o una combinación con él) para la **watchlist de
  usuarios** priorizada. El drill-down por usuario debería mostrar los tres scores en
  paralelo para dar contexto al analista.